# Jakarta facility audit
This notebook reads the real cached OSM snapshot. No API calls or synthetic records. Run from the project root or docs directory with the project virtual environment.

In [1]:
from pathlib import Path
import json, sys
ROOT = Path.cwd() if (Path.cwd() / "facility_finder").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from facility_finder.data import normalize, audit
snapshot = json.loads((ROOT / "data/raw/latest.json").read_text())
records, ingestion_stats = normalize(snapshot)
quality = audit(records, ingestion_stats, snapshot)
print("Source:", snapshot["scope"])
print("Downloaded:", snapshot["fetched_at"])
print("Normalized OSM objects:", len(records))

Source: Jakarta administrative area, ISO3166-2=ID-JK
Downloaded: 2026-09-14T13:15:32.318207+00:00
Normalized OSM objects: 5824


In [2]:
import pandas as pd
print(pd.Series(quality["by_category"], name="records").to_string())
print(pd.DataFrame(quality["completeness"]).T.to_string())

toilets              141
place_of_worship    5645
drinking_water        38
                known   total  percent
name           5393.0  5824.0    92.60
access           45.0  5824.0     0.77
opening_hours    14.0  5824.0     0.24
wheelchair       36.0  5824.0     0.62
fee              27.0  5824.0     0.46


In [3]:
print("Source integrity:", ingestion_stats)
print("Normalized duplicate IDs:", quality["normalized_duplicate_ids"])
print("Potential duplicate/proximity pairs:", quality["near_duplicate_pair_count"])
print("OSM object types:", quality["by_osm_type"])
print("Wheelchair tag values:", quality["wheelchair_values"])
saved = json.loads((ROOT / "data/processed/quality.json").read_text())
assert quality == saved
print("Audit exactly matches saved quality.json")

Source integrity: {'raw_facility_elements': 5824, 'missing_or_invalid_coordinates': 0, 'duplicate_osm_elements_removed': 0, 'rejected_identifiers': 0}
Normalized duplicate IDs: 0
Potential duplicate/proximity pairs: 104
OSM object types: {'node': 315, 'relation': 7, 'way': 5502}
Wheelchair tag values: {'unknown': 5788, 'yes': 16, 'limited': 12, 'no': 7, 'designated': 1}
Audit exactly matches saved quality.json


## Interpretation
The grain is one OSM object, not one confirmed physical facility. Missing metadata and uneven mapping limit access decisions. Proximity pairs need manual review. Retrieval and edit times are not verification times. See ../DATA_QUALITY.md for the assessment and source documentation.